# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# Section 1: Playbook Archetypes and Operational Actions
archetypes = {
    "High-Volume Erosion (Ranked Priority 1)": {
        "Condition": "High trailing impressions + declining trend index.",
        "Reason Code": "REFRESH_URGENT_CORE",
        "Action": "Immediate human-led editorial expansion. Focus on restoring missing historical subtopics or updating stale data anchors."
    },
    "Low-Authority Long-Tail Drift (Ranked Priority 2)": {
        "Condition": "Low average position + shallow trend deterioration.",
        "Reason Code": "CONSOLIDATE_OR_PRUNE",
        "Action": "Review for canonical cross-linking optimization or merge into high-performing hub articles to prevent keyword cannibalization."
    }
}

for name, meta in archetypes.items():
    print(f"=== {name} ===\nReason Code: {meta['Reason Code']}\nAction Plan: {meta['Action']}\n")

=== High-Volume Erosion (Ranked Priority 1) ===
Reason Code: REFRESH_URGENT_CORE
Action Plan: Immediate human-led editorial expansion. Focus on restoring missing historical subtopics or updating stale data anchors.

=== Low-Authority Long-Tail Drift (Ranked Priority 2) ===
Reason Code: CONSOLIDATE_OR_PRUNE
Action Plan: Review for canonical cross-linking optimization or merge into high-performing hub articles to prevent keyword cannibalization.



## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
# Section 2: Intended Scope and Limitations
intended_use = "Internal decision-support priority queue for content refreshment planning."
limits = (
    "This system is strictly diagnostic and works optimally for evergreen informational content. "
    "It loses predictive accuracy on seasonal, news-dependent, or highly volatile promotional landers "
    "where short-term spikes do not follow structural organic lifecycle decay."
)

print(f"Intended Use: {intended_use}\nLimits: {limits}")

Intended Use: Internal decision-support priority queue for content refreshment planning.
Limits: This system is strictly diagnostic and works optimally for evergreen informational content. It loses predictive accuracy on seasonal, news-dependent, or highly volatile promotional landers where short-term spikes do not follow structural organic lifecycle decay.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
# Section 3: Human-in-the-Loop Safeguards & Anti-Automation Guardrails
nogo_list = [
    "Brand core homepages, structural checkouts, and primary transactional product category pages.",
    "Pages undergoing major migration or domain restructuring workflows.",
    "Regulated or high-sensitivity policy content (e.g., medical or legal advisories)."
]

print("--- AUTOMATION NO-GO LIST (NEVER AUTOMATE CODES) ---")
for idx, item in enumerate(nogo_list, 1):
    print(f"{idx}. {item}")
print("\nRule: Machine models recommend and rank priority; humans decide context and execute changes.")

--- AUTOMATION NO-GO LIST (NEVER AUTOMATE CODES) ---
1. Brand core homepages, structural checkouts, and primary transactional product category pages.
2. Pages undergoing major migration or domain restructuring workflows.
3. Regulated or high-sensitivity policy content (e.g., medical or legal advisories).

Rule: Machine models recommend and rank priority; humans decide context and execute changes.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
# Section 4: Maintenance Loop and Drift Triggers
monitoring_plan = (
    "Retrain and recalibrate features every 30 days. "
    "A hard trigger to retrain immediately occurs if the baseline portfolio shift exceeds a "
    "15% increase in total 'down' trend tags month-over-month, signaling external landscape shocks "
    "or algorithmic tracking index re-classifications."
)

print(f"Monitoring Protocol: {monitoring_plan}")

Monitoring Protocol: Retrain and recalibrate features every 30 days. A hard trigger to retrain immediately occurs if the baseline portfolio shift exceeds a 15% increase in total 'down' trend tags month-over-month, signaling external landscape shocks or algorithmic tracking index re-classifications.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import pandas as pd
import numpy as np
import os

# 1. Dataset recovery routine
filename = "content_refresh_anonymized.csv"
df = None
for root, dirs, files in os.walk("."):
    if filename in files:
        df = pd.read_csv(os.path.join(root, filename))
        break
if df is None:
    raw_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(raw_url)

# 2. Extract Available Core Columns Dynamically
available_cols = list(df.columns)
feature_candidates = ['avg_position', 'ctr', 'word_count', 'search_volume', 'impressions_90d', 'content_age_days']
features = [col for col in feature_candidates if col in available_cols]
target_col = [col for col in ['trend_direction', 'trend', 'direction'] if col in available_cols][0]

# Ensure we have an explicit identifier, fallback to index string if missing
id_col = [col for col in ['page_id', 'id'] if col in available_cols]
id_name = id_col[0] if id_col else 'asset_index'
if id_name == 'asset_index':
    df['asset_index'] = [f"page_{i}" for i in range(len(df))]

# 3. Build a scoring heuristic for our operational priority playbook
# Score combines impressions volume with decay flag to prioritize highest-yield assets
df['decay_score'] = (df[target_col] == 'down').astype(int) * df.get('impressions_90d', 100)
playbook_queue = df.sort_values(by='decay_score', ascending=False).copy()

# 4. Map Reason Codes dynamically
def assign_reason_code(row):
    if row[target_col] == 'down' and row.get('impressions_90d', 0) > df.get('impressions_90d', 0).median():
        return "REFRESH_URGENT_CORE"
    elif row[target_col] == 'down':
        return "CONSOLIDATE_OR_PRUNE"
    return "MONITOR_STABLE"

playbook_queue['reason_code'] = playbook_queue.apply(assign_reason_code, axis=1)

# Keep a clean subset for our report
output_cols = [id_name] + features + [target_col, 'decay_score', 'reason_code']
final_export_df = playbook_queue[output_cols].head(50) # Export Top 50 recommendations

# 5. Ensure Output Directories Exist Securely
output_dir = "../../work/outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

# Export CSV and JSON models for our paper next week
csv_path = os.path.join(output_dir, "ranked_refresh_queue.csv")
json_path = os.path.join(output_dir, "playbook_summary.json")

final_export_df.to_csv(csv_path, index=False)
final_export_df.head(5).to_json(json_path, orient="records")

print("=== EXPORT CONFIRMATION SYSTEM ===")
print(f"Successfully generated and saved: {csv_path}")
print(f"Successfully generated and saved: {json_path}\n")
print("Preview of the generated Content Action Playbook Queue:")
print(final_export_df[[id_name, 'decay_score', 'reason_code']].head(5).to_string(index=False))

=== EXPORT CONFIRMATION SYSTEM ===
Successfully generated and saved: ../../work/outputs/ranked_refresh_queue.csv
Successfully generated and saved: ../../work/outputs/playbook_summary.json

Preview of the generated Content Action Playbook Queue:
asset_index  decay_score         reason_code
  page_6653       517715 REFRESH_URGENT_CORE
 page_26844       509252 REFRESH_URGENT_CORE
 page_21819       463103 REFRESH_URGENT_CORE
 page_29879       416180 REFRESH_URGENT_CORE
 page_13537       347399 REFRESH_URGENT_CORE


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.